In [ ]:
from __future__ import annotations

import math
from pprint import pprint

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch import Tensor
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, TensorDataset
import cantera as ct

from network import *
from normalisation import *
from homogeneous_reactor import *
from thermo import *
from plotting import *

torch.set_default_dtype(torch.float64)
torch.set_default_device("cpu")

torch.set_num_threads(1)

# 1) Data generation - isobaric homogeneous reactor

In [ ]:
gas = ct.Solution("h2o2.yaml")
species = gas.species_names
reactive_species = species[:-2]

print(f"species: \t\t{species}")
print(f"reactive species: \t{reactive_species}")

initial_conditions = {
    "T":980,
    "phi": 1.0,
    "p": ct.one_atm,
}

fuel = "H2: 1.0"
oxidizer = "O2: 0.21, N2: 0.79"

gas.set_equivalence_ratio(initial_conditions["phi"], fuel=fuel, oxidizer=oxidizer, basis="mole")
initial_conditions["Y"] = gas.Y.tolist()

print("\ninitial_conditions: ")
pprint(initial_conditions)

In [ ]:
# solve homogeneous reactor (traditional numerical solver)

t_span = [0, 2e-3]
dt = 1e-6

t, T, Y = solve_hr(gas, ic=initial_conditions, t_span=t_span, solver="BDF", dt_eval=dt)
PV = Y[:,gas.species_index("H2O")]

# ensure homogeneous reactor reached equilibrium
gas.TPY = initial_conditions["T"], initial_conditions["p"], initial_conditions["Y"]
gas.equilibrate("HP")

assert np.isclose(gas.T, T[-1])

fig = plot_Y_over_t(t, Y, species_names=reactive_species)
plt.show()

fig = plot_Y_over_PV(PV, Y, species_names=reactive_species)
plt.show()

# 2) Calculate increments
## integrate all thermochemical states for t = [0, dt_increment]


In [ ]:
dY = np.zeros_like(Y)
for i in range(dY.shape[0]):
    ic = {
        "T": T[i],
        "p": ct.one_atm,
        "Y": Y[i,:],
    }
    _, T_incr, Y_incr = solve_hr(gas, ic, t_span=[0, dt], solver="BDF", dt_max=dt)
    dY[i,:] = Y_incr[-1,:] - Y[i,:]

In [ ]:
# calculate temperature increment
TY = np.concat((T.reshape(-1,1), Y), axis=1)
dT = calculate_dT(torch.tensor(TY), dY, gas).detach().numpy()

plt.figure(figsize=(4,3))
plt.plot(PV, dT)
plt.ylabel("dT [K]")
plt.xlabel("PV [-]")
plt.tight_layout()
plt.show()

# 3) Define network architecture

In [ ]:
input_names = ["H2O"]
target_names = reactive_species
n_neurons = [32, 32]

input_indices = [1 + gas.species_index(name) for name in input_names]
target_indices = [1 + gas.species_index(name) for name in target_names]

inputs = torch.tensor(np.concat((T.reshape(-1,1), Y), axis=1))
targets =torch.tensor(np.concat((dT.reshape(-1,1), dY), axis=1))
x = inputs[:, input_indices]
y = targets[:, target_indices]

n_inputs = x.shape[1]
n_outputs = y.shape[1]

print(f"{n_inputs=}")
print(f"{n_neurons=}")
print(f"{n_outputs=}")

cfg = FNNConfig(
    input_dim=n_inputs,
    output_dim=n_outputs,
    hidden_dims=n_neurons,
    activation="tanh",
)

model = FNN(cfg)
print("\n", model)


# 4) Data normalisation

In [ ]:
x_scaler = RootScaler()
y_scaler = RootScaler()

x_norm = x_scaler.fit_transform(x)
y_norm = y_scaler.fit_transform(y)

# 5) Train network

In [ ]:
lr = 1e-3
epochs = 10000

loss_fn = torch.nn.MSELoss()
optimizer = Adam(model.parameters(), lr=lr)

losses = []
for e in range(1, epochs + 1):

    pred = model(x_norm)
    loss = loss_fn(pred, y_norm)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()   

    if e % max(1, epochs // 10) == 0 or e == 1:# or e == epochs:
        print(
            f"Epoch {e:>4d}/{epochs}  "
            f"train_loss={loss:.2e}"#  val_loss={val_loss:.6f}"
        )

    losses.append(loss.detach().numpy())

plot_loss(losses)

# 6) A priori analysis - prediction parity

In [ ]:
sp = "H2O"

y_pred = model(x_norm).detach().numpy()

fig = plot_parity(y_norm, y_pred, sp)
plt.show()

# 7) Coupled equations

$$
\begin{align}
\frac{\partial T}{\partial t} &= \frac{1}{c_p}\dot{\omega}_T = \frac{\Delta T}{\Delta t} \\
\frac{\partial Y_k}{\partial t} &= \dot{\omega}_k = \frac{\Delta Y_k}{\Delta t}
\end{align}
$$

In [ ]:
def coupled_rhs(TY, model):

    dTY = torch.zeros_like(TY)

    print(f"T = {TY[:,0]}")

    X_norm = x_scaler.transform(TY[:, input_indices])
    y_norm = model(X_norm)
    dTY[:, target_indices] = y_scaler.inverse_transform(y_norm)
    dTY[:, 0] = calculate_dT(TY=TY, dY=dTY[:, 1:], gas=gas)
    
    return dTY.detach().numpy()

In [ ]:
def solve_hr_ml(
        model,
        gas: ct.Solution,
        ic: dict[str, float],
        t_span: tuple[float, float],
        dt: float,
    ):

    n_steps = int((t_span[1] - t_span[0]) / dt) + 1

    n_vars = 1 + len(ic["Y"])

    t = torch.zeros((n_steps,))
    y = torch.zeros((n_steps, n_vars))

    y[0,0] = ic["T"]
    y[0,1:] = torch.tensor(ic["Y"])

    for j in range(1, n_steps):

        t[j] = t[j-1] + dt
        y[j, :] = y[j-1, :] + coupled_rhs(y[j-1, :].unsqueeze(0), model)# * dt #+ 1e-5
    
    return 1e3*t, y[:,0], y[:,1:]

# 8) A posteriori analyses

In [ ]:
t_ml, T_ml, Y_ml = solve_hr_ml(model, gas, initial_conditions, t_span, dt)

In [ ]:
fig = plot_comparison([t, t_ml], [Y, Y_ml], species_names=target_names)
plt.show()